# 05 · E2: control adherence of the generated audio

**Question.** When the controls are changed at generation time, does the generated music change as requested? E1 (notebook 04) showed that the model uses the conditions to predict real audio, and that it relies almost only on tension. E2 measures the controls on generated audio, which is what the title of the project claims.

**Design (paired).** For every prompt (1 s of a real track the model was not trained on) and seed, six conditions are generated with the same random numbers (`shared_noise=True`), so differences between them are caused by the controls only (`src/control_adherence.py`):
- a 2 × 2 factorial with constant controls: tension {low, high} × combat score {low, high}. The two controls vary independently, so their effects can be separated (in notebook 03 they changed together);
- a tension step (low → high at 5 s, combat score at its median) and a combat-score step (low → high at 5 s, tension at its median).

Levels: 10th / 50th / 90th percentiles of the Slakh training conditions (tension 0.14 / 0.41 / 0.77, combat score 0.21 / 0.60 / 0.85; notebook 02, Part A), the same for both checkpoints.

**Measurements.** Tension (frame RMS, training normalisation) and combat score (the training formula over one window) on the decoded clips, without peak normalisation, on the spans 1–10 s (`full`), 1–5 s (`pre`) and 5.5–10 s (`post`). The effect of a requested change is the paired difference of the measured values, with a bootstrap 95% CI over prompt × seed units; the gain is that difference divided by the requested difference (1 = the output changes as much as requested).

**Codec reference.** The real continuation of every prompt is decoded with the same 8 codebooks and measured in the same way, to show how well the measurements recover the stored controls when there is no generation error.

**Runtime.** About 1–1.5 h on a T4 or P100 (288 clips: 2 checkpoints × 12 prompts × 2 seeds × 6 conditions). Results are written after every prompt.

**Kaggle settings:** GPU (T4 or P100), Internet on. **Inputs:** `slakh-tensors-final`, `ost-tensors-sky-w3`, models `secunda-finetuned-on-slakh` and `secunda-finetuned-on-ost-v2`.

In [ ]:
# ── Repository bootstrap: make src/ importable on Kaggle and locally ──
# On Kaggle the repository is cloned into /kaggle/working (Internet must be
# enabled in the notebook settings). Locally, set SECUNDA_REPO_DIR to the
# repository root and nothing is cloned.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DanBracs7/Adaptive-Secunda-Synth.git"
REPO_BRANCH = "revision"  # switch to "main" after merging the revision
REPO_DIR = Path(os.environ.get("SECUNDA_REPO_DIR", "/kaggle/working/Adaptive-Secunda-Synth"))

if not (REPO_DIR / "src").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR / "src"))

commit = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO_DIR)
print("Commit:", commit or "unknown (not a git checkout)")

In [ ]:
from paths import get_path, running_on_kaggle, source_roots_for_split

if running_on_kaggle():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "encodec==0.1.1"], check=True)

In [ ]:
import csv
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import control_adherence as ca
from condition_ablation import bootstrap_mean_ci
from control_features import KNOWN_TENSION_RMS_STATS, load_norm_stats
from generation import Timer, decode_codes, generate_codes, load_codec, load_prompt, save_wav
from training_utils import load_checkpoint_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

PROMPTS_PER_CHECKPOINT = 12
SEEDS = (0, 1)
PROMPT_START_FRAME = 1500     # 20 s into the track, past most intros
SAVE_AUDIO = True             # WAV files for listening and for the listening test (E3)

NORM_STATS = load_norm_stats(get_path("norm_stats"))
# v1: statistics of the largest block of Slakh training tensors and of all
# OST tensors (notebook 02, Part B; the five versions differ by <= 0.019).
TENSION_STATS = KNOWN_TENSION_RMS_STATS["v1"]

OUTPUT_DIR = get_path("output_dir") / "control_adherence"
AUDIO_DIR = OUTPUT_DIR / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("Conditions:", ca.CONDITIONS)
print("Levels:", ca.LEVELS)

In [ ]:
# Prompts from tracks the checkpoints were not trained on: evenly spaced
# Slakh test tracks for Slakh-FT, all OST validation tracks (6 Skyrim +
# 6 Witcher 3) for OST-FT.
with open(get_path("manifest"), newline="", encoding="utf-8") as file:
    manifest_rows = list(csv.DictReader(file))


def tensor_path_of(row):
    return source_roots_for_split(row["split"])[row["source"]] / Path(row["path"]).name


def usable(row):
    return int(row["num_frames"]) >= PROMPT_START_FRAME + ca.TOTAL_FRAMES


slakh_test_rows = sorted([r for r in manifest_rows if r["source"] == "slakh" and r["split"] == "test" and usable(r)],
                         key=lambda r: r["path"])
stride = max(1, len(slakh_test_rows) // PROMPTS_PER_CHECKPOINT)
ost_val_rows = sorted([r for r in manifest_rows if r["source"] in ("skyrim", "witcher3") and r["split"] == "val" and usable(r)],
                      key=lambda r: r["path"])

SETUPS = {
    "Slakh-FT": {"checkpoint": get_path("ckpt_slakh_ft"),
                 "rows": slakh_test_rows[::stride][:PROMPTS_PER_CHECKPOINT]},
    "OST-FT": {"checkpoint": get_path("ckpt_ost_ft"),
               "rows": ost_val_rows[:PROMPTS_PER_CHECKPOINT]},
}
for name, setup in SETUPS.items():
    print(f"{name}: {len(setup['rows'])} prompts:", ", ".join(Path(r["path"]).stem for r in setup["rows"]))

## Generation and measurement

In [ ]:
codec = load_codec(device)
clip_rows, reference_rows = [], []
curves = {}                    # (checkpoint, unit, condition) -> measured tension curve
reference_curves = {}          # (checkpoint, prompt) -> (stored, measured) tension curves
started = time.time()

for checkpoint_name, setup in SETUPS.items():
    model, _ = load_checkpoint_model(setup["checkpoint"], device)

    for prompt_index, row in enumerate(setup["rows"]):
        path = tensor_path_of(row)
        real_codes, stored_tension, stored_combat = load_prompt(path, PROMPT_START_FRAME, ca.TOTAL_FRAMES)
        prompt = real_codes[:, :, :ca.PROMPT_FRAMES]
        prompt_tension = stored_tension[:ca.PROMPT_FRAMES]
        prompt_combat = stored_combat[:ca.PROMPT_FRAMES]

        # Codec reference: the real continuation, decoded and measured the same way.
        reference_wave = decode_codes(codec, real_codes)[0]
        record, curve = ca.measure_clip(reference_wave, NORM_STATS, TENSION_STATS)
        reference_rows.append({"checkpoint": checkpoint_name, "prompt": prompt_index, "track": Path(row["path"]).stem,
                               **record, **ca.requested_span_means(stored_tension, stored_combat)})
        reference_curves[(checkpoint_name, prompt_index)] = (stored_tension.numpy(), curve)

        tracks = [ca.requested_controls(c, prompt_tension, prompt_combat) for c in ca.CONDITIONS]
        tension = torch.stack([t for t, _ in tracks]).to(device)
        combat = torch.stack([c for _, c in tracks]).to(device)
        prompts = prompt.expand(len(ca.CONDITIONS), -1, -1).to(device)

        for seed in SEEDS:
            unit = f"{prompt_index:02d}_s{seed}"
            with Timer() as timer:
                codes = generate_codes(model, prompts, tension, combat, seed=seed, shared_noise=True)
            waveforms = decode_codes(codec, codes)

            for index, condition in enumerate(ca.CONDITIONS):
                record, curve = ca.measure_clip(waveforms[index], NORM_STATS, TENSION_STATS)
                clip = {"checkpoint": checkpoint_name, "unit": unit, "prompt": prompt_index, "seed": seed,
                        "track": Path(row["path"]).stem, "condition": condition,
                        "peak": float(waveforms[index].abs().max()),
                        **record, **ca.requested_span_means(tracks[index][0], tracks[index][1])}
                if SAVE_AUDIO:
                    clip["clipped_fraction"] = save_wav(AUDIO_DIR / f"{checkpoint_name}_{unit}_{condition}.wav",
                                                        waveforms[index])
                clip_rows.append(clip)
                curves[(checkpoint_name, unit, condition)] = curve

            print(f"{checkpoint_name} prompt {prompt_index + 1}/{len(setup['rows'])} seed {seed}: "
                  f"generation {timer.seconds:.0f} s, elapsed {(time.time() - started) / 60:.0f} min")

        # Write after every prompt, so a stopped session keeps its results.
        pd.DataFrame(clip_rows).to_csv(OUTPUT_DIR / "clips.csv", index=False)
        pd.DataFrame(reference_rows).to_csv(OUTPUT_DIR / "codec_reference.csv", index=False)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

clips = pd.DataFrame(clip_rows)
reference = pd.DataFrame(reference_rows)
np.savez_compressed(OUTPUT_DIR / "tension_curves.npz",
                    **{"|".join(key): value for key, value in curves.items()})
print(f"{len(clips)} clips in {(time.time() - started) / 60:.0f} min")

## A · Factorial: effect of each requested control on each measured control

In [ ]:
cell_means = (clips[clips["condition"].isin(ca.FACTORIAL_CONDITIONS)]
              .groupby(["checkpoint", "condition"])[["tension_full", "combat_full", "requested_tension_full", "requested_combat_full"]]
              .mean())
print("Mean measured values per cell (1-10 s):")
print(cell_means.round(3).to_string())

effect_tables = []
for checkpoint_name in SETUPS:
    effects = ca.factorial_effects(clips[clips["checkpoint"] == checkpoint_name])
    effects.insert(0, "checkpoint", checkpoint_name)
    effect_tables.append(effects)
factorial = pd.concat(effect_tables, ignore_index=True)
factorial.to_csv(OUTPUT_DIR / "factorial_effects.csv", index=False)
print("\nPaired effects (high - low request, averaged over the other control):")
print(factorial.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

print("\nClip-level correlation between requested and measured values (factorial clips):")
for checkpoint_name in SETUPS:
    subset = clips[(clips["checkpoint"] == checkpoint_name) & clips["condition"].isin(ca.FACTORIAL_CONDITIONS)]
    r_tension = np.corrcoef(subset["requested_tension_full"], subset["tension_full"])[0, 1]
    r_combat = np.corrcoef(subset["requested_combat_full"], subset["combat_full"])[0, 1]
    print(f"{checkpoint_name:9s} tension r={r_tension:.3f}   combat score r={r_combat:.3f}   (n={len(subset)})")

## B · Steps at 5 s

In [ ]:
steps = []
for checkpoint_name in SETUPS:
    effects = ca.step_effects(clips[clips["checkpoint"] == checkpoint_name])
    effects.insert(0, "checkpoint", checkpoint_name)
    steps.append(effects)
steps = pd.concat(steps, ignore_index=True)
steps.to_csv(OUTPUT_DIR / "step_effects.csv", index=False)
print("Post-step (5.5-10 s) minus pre-step (1-5 s) change:")
print(steps.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

print("\nLatency of the tension step (curve averaged over units, midpoint crossing):")
for checkpoint_name in SETUPS:
    units = sorted(clips[clips["checkpoint"] == checkpoint_name]["unit"].unique())
    mean_curve = np.mean([curves[(checkpoint_name, u, "tension_step")] for u in units], axis=0)
    latency = ca.step_latency_seconds(mean_curve)
    print(f"{checkpoint_name:9s} " + (f"{latency:.2f} s" if latency is not None else "no rise"))

## C · Codec reference: measurements on real audio decoded with 8 codebooks

In [ ]:
reference_summary = []
for checkpoint_name in SETUPS:
    subset = reference[reference["checkpoint"] == checkpoint_name]
    correlations, errors = [], []
    for prompt_index in subset["prompt"]:
        stored, measured = reference_curves[(checkpoint_name, prompt_index)]
        start, end = ca.SPANS["full"]
        correlations.append(np.corrcoef(stored[start:end], measured[start:end])[0, 1])
        errors.append(np.mean(np.abs(stored[start:end] - measured[start:end])))
    reference_summary.append({
        "prompts from": checkpoint_name,
        "tension frame r (median over prompts)": float(np.nanmedian(correlations)),
        "tension mean abs error": float(np.mean(errors)),
        "tension stored mean": subset["requested_tension_full"].mean(),
        "tension measured mean": subset["tension_full"].mean(),
        "combat stored mean": subset["requested_combat_full"].mean(),
        "combat measured mean": subset["combat_full"].mean(),
        "combat r across prompts": float(np.corrcoef(subset["requested_combat_full"], subset["combat_full"])[0, 1]),
    })
reference_summary = pd.DataFrame(reference_summary)
reference_summary.to_csv(OUTPUT_DIR / "codec_reference_summary.csv", index=False)
print(reference_summary.T.to_string(float_format=lambda value: f"{value:.3f}"))

## Figure: requested vs measured controls (for the report)

In [ ]:
seconds = np.arange(ca.TOTAL_FRAMES) / ca.FRAME_RATE
figure, axes = plt.subplots(1, 3, figsize=(15, 3.8))
colors = {"Slakh-FT": "tab:blue", "OST-FT": "tab:orange"}

# (a) tension over time: low vs high request (factorial, averaged over combat levels) and the step.
for checkpoint_name in SETUPS:
    units = sorted(clips[clips["checkpoint"] == checkpoint_name]["unit"].unique())
    for conditions, style, label in ((("tL_cL", "tL_cH"), ":", "low"), (("tH_cL", "tH_cH"), "--", "high")):
        mean_curve = np.mean([curves[(checkpoint_name, u, c)] for u in units for c in conditions], axis=0)
        axes[0].plot(seconds, mean_curve, style, color=colors[checkpoint_name], linewidth=1, label=f"{checkpoint_name} {label}")
    mean_step = np.mean([curves[(checkpoint_name, u, "tension_step")] for u in units], axis=0)
    axes[1].plot(seconds, mean_step, color=colors[checkpoint_name], label=checkpoint_name)
for level in ("low", "high"):
    axes[0].axhline(ca.LEVELS["tension"][level], color="grey", linewidth=0.6)
requested_step = np.where(np.arange(ca.TOTAL_FRAMES) < ca.STEP_FRAME, ca.LEVELS["tension"]["low"], ca.LEVELS["tension"]["high"])
axes[1].plot(seconds[ca.PROMPT_FRAMES:], requested_step[ca.PROMPT_FRAMES:], color="black", linestyle="--", linewidth=1, label="requested")
axes[0].set_title("(a) measured tension, constant request")
axes[1].set_title("(b) measured tension, step at 5 s")
for axis in axes[:2]:
    axis.axvline(ca.PROMPT_FRAMES / ca.FRAME_RATE, color="grey", linewidth=0.6)
    axis.set_xlabel("time (s)")
    axis.set_ylim(0, 1)
    axis.legend(fontsize=7)
axes[0].set_ylabel("tension")

# (c) paired effects with 95% CI, as gain (1 = measured change equals requested change).
labels, values, lows, highs, bar_colors = [], [], [], [], []
for checkpoint_name in SETUPS:
    subset = factorial[factorial["checkpoint"] == checkpoint_name].set_index(["requested", "measured"])
    for (requested, measured), text in ((("tension", "tension"), "T→T"), (("combat_score", "combat_score"), "C→C"),
                                        (("tension", "combat_score"), "T→C"), (("combat_score", "tension"), "C→T")):
        row = subset.loc[(requested, measured)]
        scale = ca.LEVELS[requested]["high"] - ca.LEVELS[requested]["low"]
        labels.append(f"{checkpoint_name}\n{text}")
        values.append(row["effect"] / scale)
        lows.append((row["effect"] - row["ci_low"]) / scale)
        highs.append((row["ci_high"] - row["effect"]) / scale)
        bar_colors.append(colors[checkpoint_name])
axes[2].bar(range(len(values)), values, yerr=[lows, highs], color=bar_colors, capsize=3)
axes[2].axhline(0, color="black", linewidth=0.6)
axes[2].axhline(1, color="grey", linewidth=0.6, linestyle="--")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(labels, fontsize=7)
axes[2].set_title("(c) effect / requested change (95% CI)")

figure.tight_layout()
figure.savefig(OUTPUT_DIR / "control_adherence.png", dpi=200)
figure.savefig(OUTPUT_DIR / "control_adherence.pdf")
plt.show()

### How to read the result

- **T→T** (requested tension → measured tension): a gain near 1 with a CI above 0 means the tension control works as requested.
- **C→C** (requested combat score → measured combat score): the main open question after E1. A CI that includes 0 means the combat control has no measurable effect on the generated audio.
- **T→C and C→T** (cross-talk): how much changing one control moves the other measurement. A positive T→C would mean that "combat" is obtained through tension (louder audio also scores higher on percussive loudness).
- **Steps:** the post − pre change for the stepped control, and the latency of the tension step.
- **Codec reference:** the agreement that can be expected without generation errors. Gains and correlations of generated clips should be read relative to it.